TITLE :Hotel Management System
Using SQL (SQLite) in Google Colab

DESCRIPTION: This project is a Hotel Management System developed using SQL (SQLite) and Python in Google Colab.

The system is designed to manage hotel operations including guest records, room availability, bookings, and payments. It demonstrates how a relational database can be used to efficiently store, manage, and retrieve data.

The project includes multiple interconnected tables such as Guests, Rooms, Bookings, and Payments. SQL queries are used to perform operations like retrieving available rooms, tracking bookings, and calculating total revenue.

This system provides a basic understanding of real-world database management and can be extended into a full-scale application.

In [5]:
import sqlite3
import os

# Remove existing database files if they exist to prevent 'UNIQUE constraint failed' errors
if os.path.exists('hotel.db'):
    os.remove('hotel.db')
if os.path.exists('hotel.db-journal'): # Ensure journal file is also removed
    os.remove('hotel.db-journal')

# Establish a database connection and create a cursor if they don't exist or are closed
# This assumes 'conn' and 'cursor' are not globally defined and open.
# If a connection exists, it should be re-opened or re-created.
conn = sqlite3.connect('hotel.db')
cursor = conn.cursor()
# creating tables
cursor.execute("""
CREATE TABLE IF NOT EXISTS Guests (
    guest_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    phone TEXT,
    email TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Rooms (
    room_id INTEGER PRIMARY KEY,
    room_type TEXT,
    price REAL,
    status TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Bookings (
    booking_id INTEGER PRIMARY KEY,
    guest_id INTEGER,
    room_id INTEGER,
    check_in TEXT,
    check_out TEXT,
    FOREIGN KEY(guest_id) REFERENCES Guests(guest_id),
    FOREIGN KEY(room_id) REFERENCES Rooms(room_id)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS Payments (
    payment_id INTEGER PRIMARY KEY,
    booking_id INTEGER,
    amount REAL,
    payment_date TEXT,
    FOREIGN KEY(booking_id) REFERENCES Bookings(booking_id)
)
""")
# sample data
# guest Information
cursor.execute("INSERT INTO Guests VALUES (1, 'Ali', '03001234567', 'ali@email.com')")
cursor.execute("INSERT INTO Guests VALUES (2, 'Sara', '03007654321', 'sara@email.com')")
cursor.execute("INSERT INTO Guests VALUES (3, 'Ahmed', '03111222333', 'ahmed@email.com')")
cursor.execute("INSERT INTO Guests VALUES (4, 'Ayesha', '03222333444', 'ayesha@email.com')")
# room Information
cursor.execute("INSERT INTO Rooms VALUES (1, 'Single', 5000, 'Available')")
cursor.execute("INSERT INTO Rooms VALUES (2, 'Double', 8000, 'Available')")
cursor.execute("INSERT INTO Rooms VALUES (3, 'Suite', 12000, 'Available')")
cursor.execute("INSERT INTO Rooms VALUES (4, 'Deluxe', 15000, 'Booked')")
# booking information
cursor.execute("INSERT INTO Bookings VALUES (1, 1, 4, '2026-04-01', '2026-04-05')")
# payment information
cursor.execute("INSERT INTO Payments VALUES (1, 1, 60000, '2026-04-01')")

# Commit the changes and close the connection after operations are done
conn.commit()
conn.close()

Runing First query about the available Rooms


In [8]:
import sqlite3

# Re-establish the database connection for this cell
conn = sqlite3.connect('hotel.db')
cursor = conn.cursor()

query = """
SELECT room_id, room_type, price
FROM Rooms
WHERE status = 'Available'
"""

for row in cursor.execute(query):
    print(row)

# Close the connection after use in this cell
conn.close()

(1, 'Single', 5000.0)
(2, 'Double', 8000.0)
(3, 'Suite', 12000.0)


For professional look

In [10]:
import pandas as pd
import sqlite3

# Re-establish the database connection for this cell
conn = sqlite3.connect('hotel.db')

df = pd.read_sql_query(query, conn)
display(df)

# Close the connection after use in this cell
conn.close()

,room_id,room_type,price
0,1,Single,5000.0
1,2,Double,8000.0
2,3,Suite,12000.0


Show Booking Details (JOIN)

In [12]:
import sqlite3

# Re-establish the database connection for this cell
conn = sqlite3.connect('hotel.db')

query = """
SELECT
    Guests.name AS Guest_Name,
    Rooms.room_type AS Room_Type,
    Rooms.price,
    Bookings.check_in,
    Bookings.check_out
FROM Bookings
JOIN Guests ON Guests.guest_id = Bookings.guest_id
JOIN Rooms ON Rooms.room_id = Bookings.room_id
"""

df = pd.read_sql_query(query, conn)
display(df)

# Close the connection after use in this cell
conn.close()

,Guest_Name,Room_Type,price,check_in,check_out
0,Ali,Deluxe,15000.0,2026-04-01,2026-04-05


Calculate Total Revenue

In [13]:
import sqlite3

# Re-establish the database connection for this cell
conn = sqlite3.connect('hotel.db')
query = """
SELECT SUM(amount) AS Total_Revenue
FROM Payments
"""

df = pd.read_sql_query(query, conn)
df

,Total_Revenue
0,60000.0


Count Total Guests

In [14]:
import sqlite3

# Re-establish the database connection for this cell
conn = sqlite3.connect('hotel.db')
query = """
SELECT COUNT(*) AS Total_Guests
FROM Guests
"""

df = pd.read_sql_query(query, conn)
df

,Total_Guests
0,4
